<a href="https://colab.research.google.com/github/siva2513-ship-it/paddy_disease_detection/blob/main/colab_notebooks/XGBoost_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from pathlib import Path

features_path = Path(
    "/content/drive/MyDrive/Paddy_Disease_Project/features"
)

X_train = np.load(features_path / "X_train.npy")
y_train = np.load(features_path / "y_train.npy")
X_valid = np.load(features_path / "X_valid.npy")
y_valid = np.load(features_path / "y_valid.npy")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (8326, 512)
y_train: (8326,)
X_valid: (2081, 512)
y_valid: (2081,)


In [3]:
classes = [
    'bacterial_leaf_blight',
    'bacterial_leaf_streak',
    'bacterial_panicle_blight',
    'blast',
    'brown_spot',
    'dead_heart',
    'downy_mildew',
    'hispa',
    'normal',
    'tungro'
]

In [4]:
!pip install -q xgboost

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=10,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

# Training
start_time = time.time()
xgb_model.fit(X_train, y_train)
xgb_train_time = time.time() - start_time

# Prediction
start_time = time.time()
xgb_preds = xgb_model.predict(X_valid)
xgb_inference_time = time.time() - start_time

# Evaluation
xgb_accuracy = accuracy_score(y_valid, xgb_preds)

print(f"XGBoost Training time: {xgb_train_time:.4f} seconds")
print(f"XGBoost Inference time: {xgb_inference_time:.4f} seconds")
print(f"\nXGBoost Accuracy: {xgb_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_valid,
    xgb_preds,
    target_names=classes,
    digits=4
))

XGBoost Training time: 403.4509 seconds
XGBoost Inference time: 0.1684 seconds

XGBoost Accuracy: 76.17%

Classification Report:
                          precision    recall  f1-score   support

   bacterial_leaf_blight     0.6667    0.5227    0.5860        88
   bacterial_leaf_streak     0.8305    0.6282    0.7153        78
bacterial_panicle_blight     0.8750    0.5833    0.7000        72
                   blast     0.7261    0.8118    0.7666       356
              brown_spot     0.6952    0.7684    0.7300       190
              dead_heart     0.8904    0.8535    0.8715       314
            downy_mildew     0.6813    0.5487    0.6078       113
                   hispa     0.7326    0.7706    0.7511       327
                  normal     0.7692    0.8309    0.7989       337
                  tungro     0.7665    0.7330    0.7494       206

                accuracy                         0.7617      2081
               macro avg     0.7634    0.7051    0.7277      2081
           

In [6]:
import joblib

model_path = "/content/drive/MyDrive/Paddy_Disease_Project/models"

joblib.dump(
    xgb_model,
    f"{model_path}/model_5_xgboost.pkl"
)

print("XGBoost model saved successfully.")

XGBoost model saved successfully.
